# GPU speech-to-text and speakers on Colab (WhisperX)

Runs `llm.gpu.colab_diarize` on a batch of meeting recordings from the landing volume
`/Volumes/open_navigator/open_navigator_landing/crawl` and writes `<name>.diarized.json` to
`crawl/diarized/<run_date>/` ([#297](https://github.com/getcommunityone/open-navigator/issues/297)).
Steps, secrets and cost: `web_docs/docs/deployment/colab-gpu-diarize.md`.

**A GPU run uses the owner's Colab compute units.** Run it only on the owner's explicit yes for this run, read the
usage before and after, and put the cost in the run's issue or PR (`AGENTS.md`, "GPU and paid compute").

Run the cells in order. Re-running cell 5 after a disconnect skips items already done for that run date.

In [ ]:
#@title 1. Check a GPU is attached
import shutil, subprocess

if shutil.which("nvidia-smi") is None or subprocess.run(["nvidia-smi"], capture_output=True).returncode != 0:
    raise SystemExit("No GPU attached. Runtime > Change runtime type > T4 GPU > Save, then run this cell again.")
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip())

In [ ]:
#@title 2. Get the code (packages/llm at main) and install WhisperX
import os, subprocess, sys

REPO = "/content/open-navigator"
if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "--depth", "1", "--branch", "main",
                    "https://github.com/getcommunityone/open-navigator.git", REPO], check=True)
else:
    subprocess.run(["git", "-C", REPO, "pull", "--ff-only"], check=True)
# Same WhisperX pins as the Modal image (llm.gpu.diarize_common.WHISPERX_PACKAGES).
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "whisperx>=3.8,<4", "ctranslate2>=4.6,<5",
                "requests", "databricks-sdk", "loguru", "python-dotenv"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", f"{REPO}/packages/llm"], check=True)
sys.path.insert(0, f"{REPO}/packages/llm/src")
os.environ.setdefault("LLM_USAGE_LOG_DIR", "/content/llm_usage")

from llm.gpu import colab_diarize as cd
print("llm.gpu.colab_diarize ready; landing volume", cd.LANDING_VOLUME)

In [ ]:
#@title 3. Read the secrets (values are never printed)
# Colab: the key icon in the left bar > add HF_TOKEN, DATABRICKS_HOST, DATABRICKS_TOKEN > allow this notebook.
present = {name: bool(cd.get_secret(name)) for name in cd.SECRET_NAMES}
hf = cd.get_secret("HF_TOKEN")
if hf:
    os.environ["HF_TOKEN"] = hf   # pyannote's speaker models are gated
print({k: ("set" if v else "missing") for k, v in present.items()})
if not present["HF_TOKEN"]:
    raise SystemExit("Add the HF_TOKEN secret (a read token that accepted pyannote's conditions), then rerun.")
if not (present["DATABRICKS_HOST"] and present["DATABRICKS_TOKEN"]):
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs(cd.DRIVE_ROOT, exist_ok=True)
    print("No Databricks secrets: using the Drive folder", cd.DRIVE_ROOT, "(put audio under audio/ there)")

In [ ]:
#@title 4. Usage before
#@markdown Read the compute-unit balance in Colab's **Resources** panel (top right, RAM/Disk > View resources) and
#@markdown the rate it shows for this GPU, and type them here.
cu_balance_before = 0.0  #@param {type:"number"}
cu_per_hour = 0.0  #@param {type:"number"}
from datetime import datetime, timezone
from llm.usage.gpu import gpu_names

GPU = cd.gpu_kind((gpu_names() or [None])[0])
print(f"usage before: {datetime.now(timezone.utc).isoformat(timespec='seconds')} gpu={GPU} "
      f"cu_balance={cu_balance_before or 'not entered'} cu_per_hour={cu_per_hour or cd.COLAB_CU_PER_HOUR.get(GPU)}")

In [ ]:
#@title 5. Run the batch
#@markdown Items: landing-volume paths (`crawl/audio/<file>`) or direct media URLs, comma separated.
items = ""  #@param {type:"string"}
limit = 3  #@param {type:"integer"}
model_size = "small"  #@param ["small", "medium", "large-v3"]
#@markdown Leave blank for today (UTC). To resume a run from an earlier day, enter that day.
run_date = ""  #@param {type:"string"}
owner_said_yes_to_this_run = False  #@param {type:"boolean"}

if not owner_said_yes_to_this_run:
    raise SystemExit("A GPU run needs the owner's explicit yes for this run. Tick the box once you have it.")
if not items.strip():
    raise SystemExit("Enter at least one item.")
RUN_DATE = run_date.strip() or cd.today_utc()
argv = ["--items", items, "--limit", str(limit), "--model-size", model_size, "--run-date", RUN_DATE]
if cu_per_hour:
    argv += ["--cu-per-hour", str(cu_per_hour)]
exit_code = cd.main(argv)
print("results in", f"{cd.LANDING_VOLUME}/{cd.output_dir_rel(RUN_DATE)}/", "| exit code", exit_code)

In [ ]:
#@title 6. Usage after (record the cost)
cu_balance_after = 0.0  #@param {type:"number"}
row = cd.record_measured_cu(cd.make_store(), RUN_DATE, cu_balance_before or None, cu_balance_after or None, GPU)
measured = f"{row['compute_units']} CU measured" if row else "no measured CU (enter both balances)"
print(f"usage after: {datetime.now(timezone.utc).isoformat(timespec='seconds')} gpu={GPU} "
      f"cu_balance={cu_balance_after or 'not entered'}; {measured}")
print("Put this line and the summary table above in the run's issue or PR.")